# 03 — Modeling & Fair Comparison

Baseline → XGBoost (3 loss) → LightGBM → LSTM → uji ensemble. **Split & metrik identik**;
semua dievaluasi pada *common test set* (loop lengkap, tempat Loop MAE terdefinisi), dalam
detik (`expm1`). Logika di `src/models.py`.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from src.data import load_raw, add_gap_flag, clean_outliers, time_split, sequence_ready
from src import features as F, models as Mdl, metrics as M
import torch; torch.set_num_threads(8)

clean, _ = clean_outliers(add_gap_flag(load_raw()), verbose=False)
feat = F.build_base_features(clean)
tr, te = time_split(feat)
art = F.fit_feature_artifacts(tr); tr = F.transform_with_artifacts(tr, art); te = F.transform_with_artifacts(te, art)
te_c = sequence_ready(te)
Xtr, ytr_log, _ = Mdl.make_xy(tr); Xc, _, yc = Mdl.make_xy(te_c)
print(f"train={len(tr)} rows | common test = {te_c['no_do'].nunique()} loops")

train=258590 rows | common test = 2327 loops


## Baseline + tabular (XGBoost 3 objective + LightGBM)
Loss diuji di ruang log: MSE (≈ relative error), Huber (robust), MAE (L1).

In [2]:
rows = [Mdl.evaluate("Baseline (seg,hour mean)", yc, F.predict_baseline(te_c, art["baseline_model"]), te_c)]
for obj, label, extra in [("reg:squarederror", "XGBoost (MSE-log)", {}),
                          ("reg:pseudohubererror", "XGBoost (Huber-log)", {"huber_slope": 0.5, "base_score": float(ytr_log.mean())}),
                          ("reg:absoluteerror", "XGBoost (MAE-log)", {})]:
    m = Mdl.train_xgb(Xtr, ytr_log, objective=obj, **extra)
    rows.append(Mdl.evaluate(label, yc, Mdl.predict_sec(m, Xc), te_c))
ml = Mdl.train_lgbm(Xtr, ytr_log)
rows.append(Mdl.evaluate("LightGBM (MSE-log)", yc, Mdl.predict_sec(ml, Xc), te_c))
Mdl.compare_table(rows)

,Model,MAE_s,RMSE_s,MAPE_seg_%,LoopMAE_s,n_loops
0,XGBoost (MAE-log),38.11,113.89,30.36,328.78,2325
1,XGBoost (Huber-log),38.74,115.25,29.66,333.95,2325
2,LightGBM (MSE-log),38.89,115.58,28.86,340.12,2325
3,XGBoost (MSE-log),39.10,116.68,28.91,344.42,2325
4,"Baseline (seg,hour mean)",50.79,125.29,46.12,510.34,2325


## LSTM (PyTorch; sekuens per `no_do`, hanya loop lengkap)
TF 2.16/keras 3 crash di NumPy 2.x → PyTorch (D0). Loop pendek (~20 langkah) → sinyal tipis.

In [3]:
tr_c = sequence_ready(tr)
Xcols = tr_c[F.FEATURE_COLS].copy(); Xcols["is_gap_suspected"] = Xcols["is_gap_suspected"].astype(int)
scaler = StandardScaler().fit(Xcols.to_numpy(float))
max_len = Mdl.lstm_max_len(tr_c, te_c)
Xl, yl, ll, _ = Mdl.build_lstm_arrays(tr_c, F.FEATURE_COLS, max_len, scaler)
lstm = Mdl.train_lstm(Mdl.build_lstm_model(Xl.shape[2]), Xl, yl, ll, epochs=12, verbose=False)
d_te, lstm_te = Mdl.lstm_predict_sec(lstm, te_c, F.FEATURE_COLS, max_len, scaler)
rows.append(Mdl.evaluate("LSTM (Huber-log)", d_te["traveling_time_sec"].values, lstm_te, d_te))
Mdl.compare_table(rows)

,Model,MAE_s,RMSE_s,MAPE_seg_%,LoopMAE_s,n_loops
0,XGBoost (MAE-log),38.11,113.89,30.36,328.78,2325
1,XGBoost (Huber-log),38.74,115.25,29.66,333.95,2325
2,LightGBM (MSE-log),38.89,115.58,28.86,340.12,2325
3,XGBoost (MSE-log),39.10,116.68,28.91,344.42,2325
4,LSTM (Huber-log),40.23,119.57,31.09,353.86,2325
5,"Baseline (seg,hour mean)",50.79,125.29,46.12,510.34,2325


## Uji ensemble XGB+LSTM (bobot di-fit di VALIDATION, bukan test)
Stacking/weighted hanya sah bila base saling melengkapi. Di sini error berkorelasi.

In [4]:
vcut = pd.Timestamp("2026-02-19"); ls = tr.groupby("no_do")["departure_time"].transform("min")
base_tr, val = tr[ls < vcut], tr[ls >= vcut]; val_c = sequence_ready(val)
Xb, yb, _ = Mdl.make_xy(base_tr); xgb_b = Mdl.train_xgb(Xb, yb, objective="reg:absoluteerror")
btr_c = sequence_ready(base_tr)
Xc2 = btr_c[F.FEATURE_COLS].copy(); Xc2["is_gap_suspected"] = Xc2["is_gap_suspected"].astype(int)
sc = StandardScaler().fit(Xc2.to_numpy(float)); mln = Mdl.lstm_max_len(btr_c, val_c, te_c)
Xl2, yl2, ll2, _ = Mdl.build_lstm_arrays(btr_c, F.FEATURE_COLS, mln, sc)
lstm_b = Mdl.train_lstm(Mdl.build_lstm_model(Xl2.shape[2]), Xl2, yl2, ll2, epochs=12, verbose=False)
dv, lv = Mdl.lstm_predict_sec(lstm_b, val_c, F.FEATURE_COLS, mln, sc)
dt, lt = Mdl.lstm_predict_sec(lstm_b, te_c, F.FEATURE_COLS, mln, sc)
xv = Mdl.predict_sec(xgb_b, Mdl.make_xy(dv)[0]); xt = Mdl.predict_sec(xgb_b, Mdl.make_xy(dt)[0])
lmf = lambda d, p: M.loop_mae(d.assign(_p=p), "traveling_time_sec", "_p")[0]
w = float(min(np.linspace(0, 1, 21), key=lambda w: lmf(dv, w * xv + (1 - w) * lv)))
yt = dt["traveling_time_sec"].values
print(f"bobot dari validation: XGB w={w:.2f}")
Mdl.compare_table([Mdl.evaluate("XGBoost (base_tr)", yt, xt, dt),
                   Mdl.evaluate("LSTM (base_tr)", yt, lt, dt),
                   Mdl.evaluate(f"Ensemble convex (w={w:.2f})", yt, w * xt + (1 - w) * lt, dt)])

bobot dari validation: XGB w=0.65


,Model,MAE_s,RMSE_s,MAPE_seg_%,LoopMAE_s,n_loops
0,Ensemble convex (w=0.65),38.63,113.94,30.71,331.13,2325
1,XGBoost (base_tr),38.69,115.35,31.03,336.30,2325
2,LSTM (base_tr),40.44,120.54,31.13,357.74,2325


## Kesimpulan (D7/D8)
**XGBoost (MAE-log) = terbaik, Loop MAE 328.8s (lift 35.6%).** LSTM lebih lemah (loop
pendek + error berkorelasi). Ensemble hanya ~1.5% di atas base-nya & **kalah** dari XGBoost
full-train → **pilih XGBoost tunggal**: latensi 0.85 ms/loop, 1 artifact `.json`, retrain detik.